# 🎯 [TRN02_M01] 방공 식별 및 대드론 교전 수칙 퀴즈 문항 데이터베이스 및 평가 기준 수립
### Google Colab 실습 환경 (교육·훈련 / Level 2)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/TRN02/trn02_m01_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 교육·훈련 (Level 2) |
| **미션 ID** | TRN02_M01 |
| **미션 주제** | 방공 식별 및 대드론 교전 수칙 퀴즈 문항 데이터베이스 및 평가 기준 수립 |
| **소요 시간** | 약 1.5시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
미국-이란 전쟁 사례에서 보듯 드론과 미사일이 섞여 날아오는 전장에서는 장병 한 사람 한 사람이 비행체 형상을 식별하고, 아군·민간기와 구분하며, 정해진 보고·통제 절차를 지키는 능력이 방공의 기초가 된다. 그러나 부대 단위 방공 식별 교육은 채점과 오답 분석에 교관의 손이 많이 들어 개인별 약점을 제때 짚어 주기 어렵다. 본 프로젝트에서는 교육용 가상 문항 은행과 가상 응시자 응답을 코드로 만들어, 문항 데이터베이스 정의, 응답 정제·파싱, 자동 채점, 오답 유형(개념 혼동·피아 오인·절차 누락) 분류, 취약 영역 추천, 교수자 대시보드, 교관 검토(HITL), 대규모 동시 응시 부하 시험, AAR 문항 분석과 재보정, 최종 통합까지 채점·피드백 체계를 단계별로 구현한다. 최종 목표는 채점은 빠르고 정확하게 자동화하되, 이의 제기와 서술형 판단은 교관이 확정하는 신뢰할 수 있는 방공 교육 평가 도구를 만드는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

## 📂 2. 산출물 디렉토리 체계 및 실습 가이드

본 과제의 미션별 산출물(`deliverables`)은 단순 스크립트를 넘어 실무 엔터프라이즈 및 국방 교육·훈련 평가체계에 즉시 배포할 수 있는 **표준 패키지 디렉토리 구조**를 따릅니다:

| 디렉토리 | 주요 산출물(Deliverables) | 역할 및 설명 |
| :--- | :--- | :--- |
| **`data/`** | `data/schema/trn02_input_schema.json` | 문항 데이터베이스 스키마 및 가상 응답 데이터셋 규격 |
| **`src/`** | `src/preprocess/`, `src/models/`, `src/verification/`, `src/ui/`, `src/hitl/`, `src/main_trn02.py` | 응답 전처리, 자동 채점 Baseline, 오답 유형 분류, 취약 영역 추천, 교수자 대시보드, HITL 등 핵심 파이썬 소스 코드 |
| **`docs/`** | `docs/trn02_data_spec.md`, `docs/trn02_hitl_guide.md`, `docs/trn02_aar_report.md`, `docs/trn02_final_report.md` | 데이터 규격서, HITL 교관 검토 가이드, AAR 분석 보고서, 최종 운영 매뉴얼 등 기술/행정 문서 |
| **`configs/`**| `configs/trn02_tuned_params.yaml` | AAR 피드백 기반 재조정 변별도 파라미터 및 가중치 설정 파일 |
| **`tests/`** | `tests/trn02_stress_test.py` | 전 부대원 동시 응시 시나리오 대규모 트래픽 부하 테스트 및 최적화 스위트 |

> **💡 [Google Colab / 주피터 노트북 실습 팁]**
> - **방법 A. 자동 파일 저장 (권장):** 코드 셀 첫 줄에 `%%writefile src/...`와 같이 매직 명령어를 추가하여 실행하면, 노트북에서 작업한 코드가 해당 폴더의 파일로 자동 추출·저장됩니다.
> - **방법 B. 논리적 모듈 대응:** 별도 파일 추출 없이 노트북 셀 단위로 순차 실행하더라도 각 미션이 해당 모듈의 역할을 1:1로 수행하며, **미션 10(시스템 통합)**에서 전체 파이프라인이 유기적으로 완성되도록 설계되어 있습니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 01. 방공 식별 및 대드론 교전 수칙 퀴즈 문항 데이터베이스 및 평가 기준 수립
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

방공 식별 및 대드론 교전 수칙 퀴즈 문항 데이터베이스 및 평가 기준을 수립한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **형상식별·피아식별·경보보고·교전통제·대드론일반 5개 영역의 퀴즈 문항 데이터베이스 구조와 난이도별 배점·합격 기준을 정하고, 잘못된 문항·응답 기록을 걸러내는 검증기를 만든다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **선다형·단답형·순서형 문항의 필드(문항ID, 영역, 난이도, 정답, 오답 보기별 오답 유형)와 응답 기록 필드(응시자, 문항, 원본 응답, 제출 시각), 응시 상황 필드(장소·통신 상태 등) 정의하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `data/schema/trn02_input_schema.json 및 docs/trn02_data_spec.md` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 교육·훈련 실무 및 미국-이란 전쟁 전장 환경과의 군사적 부합성

#### 미션 PASS 조건

- 정량 기준: 스키마 필드 정의 완성도 100%, 샘플 데이터 검증 오류 0건


- 진행 기준: 입력 데이터 유효성 검증 스크립트 통과 및 명세서 문서화 완료


- 검증 방법: JSON Schema 검증 도구 및 샘플 데이터 파싱 단위 테스트


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""방공 식별·대드론 교전 통제 원칙 퀴즈의 문항 데이터베이스와 응답 기록 스키마, 난이도별 배점·합격 기준을 정의하고, 가상 문항·응답 샘플을 검증한다."""
import copy
import re
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# [교육용가상자료] 방공·드론 식별 퀴즈 문항 은행과 가상 응시자 응답 생성기
# 문항은 일반 원칙 수준의 가상 문항이며 실제 교범·교전규칙 원문이 아니다. 응시자는 가명(S001~)이다.
# ---------------------------------------------------------------------------
AREAS = ["형상식별", "피아식별", "경보보고", "교전통제", "대드론일반"]
ETYPES = ["개념 혼동", "피아 오인", "절차 누락"]
POINTS = {1: 2, 2: 3, 3: 5}
C, F, P = ETYPES


def _mc(qid, area, diff, stem, opts, ans, dist):
    return {"qid": qid, "type": "MC", "area": area, "diff": diff, "stem": stem, "options": opts, "answer": ans,
            "dist": dict(zip([i for i in range(1, 5) if i != ans], dist))}


def _sa(qid, area, diff, stem, accept, wrong):
    return {"qid": qid, "type": "SA", "area": area, "diff": diff, "stem": stem, "accept": accept, "wrong": wrong}


def _order(qid, area, diff, stem):
    return {"qid": qid, "type": "ORDER", "area": area, "diff": diff, "stem": stem, "answer": "ABCD"}


BANK = [
    _mc("Q01", "형상식별", 1, "꼬리날개 없이 삼각형 날개를 가진 소형 고정익 실루엣은?", ["쿼드콥터형", "삼각익 고정익형", "회전익 헬기형", "조류"], 2, [C, C, F]),
    _mc("Q02", "형상식별", 1, "회전날개 4개가 십자로 배치된 실루엣은?", ["고정익 드론형", "쿼드콥터형", "순항미사일형", "탄도미사일형"], 2, [C, C, C]),
    _mc("Q03", "형상식별", 2, "가늘고 긴 동체 중간에 작은 날개가 붙은 실루엣은?", ["순항미사일형", "탄도미사일형", "민간 경비행기형", "조류"], 1, [C, F, F]),
    _mc("Q04", "형상식별", 3, "원거리라 실루엣 식별이 어려울 때 먼저 할 행동은?", ["추정 유형으로 확정 보고", "관측 정보를 그대로 보고하고 추가 관측", "관측 중단", "민간기로 간주"], 2, [P, P, F]),
    _mc("Q05", "피아식별", 2, "식별 신호 응답이 없는 항공기에 대한 올바른 판단은?", ["즉시 적기로 확정", "미확인 항공기로 분류하고 추가 확인", "아군기로 간주", "무시"], 2, [F, F, P]),
    _mc("Q06", "피아식별", 2, "운항 정보와 일치하는 저속 비행체의 분류는?", ["적 드론", "민간기 가능성 높음, 추가 확인", "순항미사일", "아군 정찰기로 확정"], 2, [F, C, F]),
    _mc("Q07", "피아식별", 3, "아군 드론 운용 공지가 있는 공역에서 미확인 드론을 보면 우선 확인할 것은?", ["기상 정보", "아군 운용 계획과 식별 신호", "대응 준비 상태", "주변 지형"], 2, [C, P, C]),
    _mc("Q08", "피아식별", 3, "조류 무리와 소형 드론 군집 구분에 가장 도움이 되는 정보는?", ["크기만 비교", "궤적의 규칙성과 열 신호", "색상", "관측 시각"], 2, [C, C, C]),
    _mc("Q09", "경보보고", 1, "공중 위협 최초 보고에 포함하지 않아도 되는 것은?", ["관측 시각", "방향과 거리", "추정 유형", "관측자 개인 연락처"], 4, [P, P, P]),
    _mc("Q10", "경보보고", 2, "경보 전파 후 상황이 바뀌면?", ["최초 보고 유지", "변경 사항을 즉시 추가 보고", "상황 종료 후 일괄 보고", "개인 판단으로 경보 해제"], 2, [P, P, P]),
    _mc("Q11", "경보보고", 2, "보고 내용이 불확실할 때 올바른 표현은?", ["확정 표현으로 보고", "'추정'임을 밝히고 근거와 함께 보고", "보고 보류", "다른 관측자 보고를 복사"], 2, [C, P, P]),
    _mc("Q12", "경보보고", 1, "드론 탐지 후 보고 경로로 알맞은 것은?", ["인접 부대에만 전파", "지휘계통을 따라 보고", "개인 메신저로 공유", "보고 없이 대응"], 2, [P, P, P]),
    _mc("Q13", "교전통제", 1, "AI 탐지 결과만으로 대응을 결정할 수 있는가?", ["가능하다", "인간 지휘관의 승인이 필요하다", "신뢰도 90% 이상이면 가능", "야간에만 가능"], 2, [C, C, C]),
    _mc("Q14", "교전통제", 2, "민간 피해 가능성이 확인되면?", ["그대로 진행", "즉시 보고하고 지휘관 판단을 받는다", "조치 후 보고", "무시"], 2, [P, P, F]),
    _mc("Q15", "교전통제", 3, "식별이 불확실한 표적에 대한 원칙은?", ["위협으로 간주해 즉시 대응", "식별 확인 전까지 승인 절차를 따른다", "무조건 무시", "다수결로 결정"], 2, [F, F, C]),
    _mc("Q16", "교전통제", 2, "판단 근거를 기록해야 하는 이유는?", ["형식상 필요", "사후 검토와 책임 추적", "보고 생략", "기록 불필요"], 2, [C, P, P]),
    _mc("Q17", "대드론일반", 1, "대드론 체계의 일반적 처리 순서는?", ["대응-탐지-식별", "탐지-식별-보고·판단-대응", "식별-대응-탐지", "보고-탐지-대응"], 2, [P, P, P]),
    _mc("Q18", "대드론일반", 3, "레이더만으로 조류와 드론 구분이 어려운 이유는?", ["반사 신호 크기가 비슷할 수 있어서", "레이더는 야간에 못 써서", "드론에는 금속이 없어서", "조류는 탐지되지 않아서"], 1, [C, C, C]),
    _mc("Q19", "대드론일반", 2, "전자광학(EO) 센서의 약점이 커지는 조건은?", ["맑은 주간", "안개·야간", "저속 표적", "근거리"], 2, [C, C, C]),
    _mc("Q20", "대드론일반", 2, "경보가 반복해서 오경보로 판명되면?", ["경보 장치를 끈다", "사례를 기록·보고하고 기준 재검토 요청", "무시한다", "모두 실제 위협으로 간주"], 2, [P, P, F]),
    _sa("Q21", "피아식별", 1, "아군·적·민간을 구분하는 절차를 무엇이라 하는가?", ["피아식별", "IFF"], {"표적획득": C, "적기판정": F, "경계": C}),
    _sa("Q22", "형상식별", 2, "꼬리날개 없는 삼각형 날개 형태를 무엇이라 하는가?", ["삼각익", "델타익", "delta"], {"직선익": C, "후퇴익": C, "쿼드콥터": C}),
    _sa("Q23", "경보보고", 2, "불확실한 관측을 보고할 때 붙이는 표현은?", ["추정"], {"확정": C, "확인": C, "생략": P}),
    _sa("Q24", "교전통제", 1, "AI 탐지 결과에 대해 최종 결정을 내리는 주체는?", ["지휘관"], {"AI": C, "운용병": P, "레이더": C}),
    _sa("Q25", "대드론일반", 1, "조류를 드론으로 잘못 경보하는 것을 무엇이라 하는가?", ["오경보", "오탐", "false alarm"], {"미탐": C, "재경보": C, "적기": F}),
    _sa("Q26", "피아식별", 2, "운항 정보가 확인된 여객기는 어떻게 분류하는가?", ["민간기", "민항기"], {"적기": F, "미확인기": P, "아군기": F}),
    _sa("Q27", "형상식별", 1, "회전날개 4개를 가진 드론 형태는?", ["쿼드콥터", "quadcopter"], {"헥사콥터": C, "고정익": C, "헬기": F}),
    _sa("Q28", "경보보고", 1, "보고는 무엇을 따라 올라가는가?", ["지휘계통"], {"인접부대": P, "개인메신저": P, "통신망": C}),
    _order("Q29", "경보보고", 2, "최초 보고 순서: A 관측 시각, B 방향·거리, C 추정 유형, D 관측자 조치"),
    _order("Q30", "대드론일반", 2, "처리 순서: A 탐지, B 식별, C 보고·판단, D 대응"),
    _order("Q31", "교전통제", 3, "통제 순서: A 식별 확인, B 상급 승인 요청, C 승인 수령, D 조치 기록"),
    _order("Q32", "피아식별", 3, "확인 순서: A 운항정보 대조, B 식별 신호 확인, C 미확인 분류, D 보고"),
]
QIDX = {q["qid"]: q for q in BANK}
UNITS = ["가상1중대", "가상2중대", "가상3중대"]
_CIRC = "①②③④"
PARA = {"Q21": "아군적군 구분", "Q22": "세모 날개", "Q23": "추정됨으로 표기", "Q24": "중대장(지휘관)", "Q25": "잘못된 경보",
        "Q26": "민간 여객기", "Q27": "4발 드론", "Q28": "지휘 라인"}          # 뜻은 맞지만 인정 답안 목록에 없는 표현


def make_soldiers(n_per_unit, seed):
    rng = np.random.default_rng(seed)
    out = []
    for u, unit in enumerate(UNITS):
        base = [0.3, -0.1, 0.0][u]
        for i in range(n_per_unit):
            off = rng.normal(0, 0.5, len(AREAS)) + (np.array([0, 0, 0, -0.6, 0]) if u == 1 else 0)
            out.append({"sid": f"S{len(out) + 1:03d}", "unit": unit, "theta": dict(zip(AREAS, base + rng.normal(0, 0.6) + off)),
                        "tend": dict(zip(ETYPES, rng.dirichlet([1.2, 1.2, 1.2])))})
    return out


def p_correct(theta, diff):
    return 1 / (1 + np.exp(-1.7 * (theta - (diff - 2) * 0.9 + 0.9)))


def _typo(s, rng):
    if len(s) < 2:
        return s
    i = int(rng.integers(len(s) - 1))
    return s[:i] + s[i + 1] + s[i] + s[i + 2:] if len(s) <= 3 or rng.random() < 0.5 else s[:i] + s[i + 1:]


def _fmt_mc(k, rng):
    r = rng.random()
    if r < 0.55:
        return str(k)
    return rng.choice([_CIRC[k - 1], f"{k}번", "ABCD"[k - 1], "abcd"[k - 1] + ")", f" {k} ", chr(0xFF10 + k), f"{k}."])


def _fmt_sa(s, rng):
    r = rng.random()
    if r < 0.45:
        return s
    if r < 0.6:
        return " ".join(s) if len(s) <= 4 else s[:2] + " " + s[2:]
    if r < 0.72:
        return s + rng.choice(["입니다", "이다", ".", "임"])
    if r < 0.8:
        return s.upper() if s.isascii() else "".join(chr(ord(ch) + 0xFEE0) if "!" <= ch <= "~" else ch for ch in s) + " "
    return _typo(s, rng)


def answer(q, sol, rng):
    """한 문항 응답: (원본 응답 문자열, 의도한 정답 여부, 의도한 답, 오답 유형)."""
    ok = rng.random() < p_correct(sol["theta"][q["area"]], q["diff"])
    if q["type"] == "MC":
        if ok:
            k, et = q["answer"], None
        else:
            ks = list(q["dist"]); w = np.array([sol["tend"][q["dist"][k]] for k in ks])
            k = int(rng.choice(ks, p=w / w.sum())); et = q["dist"][k]
        return _fmt_mc(k, rng), ok, str(k), et
    if q["type"] == "SA":
        if ok and rng.random() < 0.06:
            return PARA[q["qid"]], ok, PARA[q["qid"]], None
        if ok:
            s, et = str(rng.choice(q["accept"])), None
        else:
            ws = list(q["wrong"]); w = np.array([sol["tend"][q["wrong"][x]] for x in ws])
            s = str(rng.choice(ws, p=w / w.sum())); et = q["wrong"][s]
        return _fmt_sa(s, rng), ok, s, et
    if ok:
        s = "ABCD"
    elif rng.random() < 0.5:
        s = "ABCD".replace("ABCD"[rng.integers(1, 4)], "")       # 단계 하나 빠뜨림
    else:
        i = int(rng.integers(0, 3)); s = list("ABCD"); s[i], s[i + 1] = s[i + 1], s[i]; s = "".join(s)   # 순서 뒤바뀜
    raw = s if rng.random() < 0.6 else rng.choice(["-".join(s), " ".join(s), s.lower(), ",".join(s)])
    return raw, ok, s, (None if ok else P)


def make_responses(n_per_unit=40, seed=7, blank=0.02):
    """응시자 x 문항 응답표(DataFrame)를 만든다. truth_* 열은 교관 기준 정답(평가용)이다."""
    rng = np.random.default_rng(seed)
    rows = []
    for sol in make_soldiers(n_per_unit, seed):
        for q in BANK:
            raw, ok, intended, et = answer(q, sol, rng)
            if rng.random() < blank:
                raw, ok, intended, et = "", False, "", None
            rows.append({"rid": f"R{len(rows):05d}", "sid": sol["sid"], "unit": sol["unit"], "qid": q["qid"], "raw": raw,
                         "truth_correct": ok, "truth_answer": intended, "truth_etype": et})
    return pd.DataFrame(rows)
# ---------------------------------------------------------------------------

# 1) 스키마: 문항(유형·영역·난이도·정답·오답 유형 매핑·상황 조건) / 응답(응시자 가명·문항·원본 응답·제출 시각)
SCHEMA = {
    "item": {
        "qid": {"type": str, "pattern": r"^Q\d{2}$"},
        "type": {"type": str, "enum": ["MC", "SA", "ORDER", "ESSAY"]},
        "area": {"type": str, "enum": AREAS},
        "diff": {"type": int, "enum": [1, 2, 3]},
        "stem": {"type": str, "minlen": 5},
    },
    "context": {                                    # 보완한 전술 상황 파라미터
        "time": {"type": str, "enum": ["주간", "야간"]},
        "weather": {"type": str, "enum": ["맑음", "안개", "비", "눈"]},
        "comm": {"type": str, "enum": ["정상", "지연", "두절"]},
        "position_error_m": {"type": (int, float), "min": 0, "max": 500},
    },
    "response": {
        "rid": {"type": str, "pattern": r"^R\d{5}$"},
        "sid": {"type": str, "pattern": r"^S\d{3}$"},
        "qid": {"type": str, "pattern": r"^Q\d{2}$"},
        "raw": {"type": str},
        "submitted_s": {"type": (int, float), "min": 0, "max": 7200},
    },
}
RUBRIC = {"배점(난이도 1/2/3)": "2 / 3 / 5점", "서술형": "10점(교관 채점)", "합격 기준": "총점의 70% 이상", "미응답": "0점, 미응답으로 기록(삭제 안 함)"}


def check(name, v, r):
    if not isinstance(v, r["type"]) or isinstance(v, bool):
        return [f"{name}: 자료형 오류({type(v).__name__})"]
    e = []
    if "enum" in r and v not in r["enum"]:
        e.append(f"{name}: 허용되지 않은 값 '{v}'")
    if "pattern" in r and not re.match(r["pattern"], v):
        e.append(f"{name}: 형식 불일치 '{v}'")
    if "minlen" in r and len(v) < r["minlen"]:
        e.append(f"{name}: 길이 부족")
    if "min" in r and v < r["min"]:
        e.append(f"{name}: 최소값 미만")
    if "max" in r and v > r["max"]:
        e.append(f"{name}: 최대값 초과")
    return e


def validate_item(it):
    e = []
    for g in ("item", "context"):
        src = it if g == "item" else it.get("context", {})
        for k, r in SCHEMA[g].items():
            e += [f"{g}.{k}: 필수 필드 누락"] if k not in src else check(f"{g}.{k}", src[k], r)
    if it.get("type") == "MC":
        if len(it.get("options", [])) != 4 or it.get("answer") not in (1, 2, 3, 4):
            e.append("MC: 보기 4개·정답 번호(1~4) 필요")
        elif set(it.get("dist", {})) != {1, 2, 3, 4} - {it["answer"]} or not set(it["dist"].values()) <= set(ETYPES):
            e.append("MC: 오답 보기별 오답 유형 매핑 불완전")
    if it.get("type") == "SA" and (not it.get("accept") or not set(it.get("wrong", {}).values()) <= set(ETYPES)):
        e.append("SA: 인정 답안·오답 유형 목록 필요")
    return e


def validate_resp(r):
    e = []
    for k, rule in SCHEMA["response"].items():
        e += [f"response.{k}: 필수 필드 누락"] if k not in r else check(f"response.{k}", r[k], rule)
    if r.get("qid") not in QIDX:
        e.append("response.qid: 문항 DB에 없는 문항")
    return e


rng = np.random.default_rng(1)
items = []
for q in BANK:
    it = copy.deepcopy(q)
    it["context"] = {"time": str(rng.choice(["주간", "야간"])), "weather": str(rng.choice(["맑음", "안개", "비", "눈"])),
                     "comm": str(rng.choice(["정상", "지연"])), "position_error_m": float(round(rng.uniform(5, 80), 1))}
    items.append(it)
resp = make_responses(5, seed=3).head(50)
recs = [{"rid": r.rid, "sid": r.sid, "qid": r.qid, "raw": r.raw, "submitted_s": float(i * 12)} for i, r in enumerate(resp.itertuples())]
ni, nr = sum(not validate_item(i) for i in items), sum(not validate_resp(r) for r in recs)
print(f"[교육용가상자료] 문항 {len(items)}개 중 검증 통과 {ni}개, 응답 {len(recs)}건 중 통과 {nr}건 (목표 오류 0건: {'달성' if ni == len(items) and nr == len(recs) else '미달'})")

bad = copy.deepcopy(items[0]); bad["qid"] = "Q1"; bad["diff"] = 4; bad["dist"].pop(4); del bad["context"]["comm"]
bad_r = dict(recs[0], sid="병사1", qid="Q99", submitted_s=9999)
print("\n[오류 주입 결과]")
for e in validate_item(bad) + validate_resp(bad_r):
    print(" -", e)

print("\n[문항 구성]")
print(pd.crosstab(pd.Series([i["area"] for i in items], name="영역"), pd.Series([i["type"] for i in items], name="유형"), margins=True, margins_name="합계"))
print("\n[평가 기준]")
for k, v in RUBRIC.items():
    print(f" - {k}: {v}")
print(f" - 만점: {sum(POINTS[i['diff']] for i in items)}점(서술형 제외), 오답 유형: {', '.join(ETYPES)}")
n_fields = sum(len(v) for v in SCHEMA.values())
print(f"\n스키마 필드 {n_fields}개 정의(문항 {len(SCHEMA['item'])}, 상황 {len(SCHEMA['context'])}, 응답 {len(SCHEMA['response'])}), 정의 완성도 100%")
print("문항 예시:", items[3]["qid"], items[3]["stem"], "| 정답", items[3]["answer"], "| 오답 유형", items[3]["dist"])


<details>
<summary><b>💡 [참고] 미션 01 예상 실행 결과 (Output)</b></summary>

```text
[교육용가상자료] 문항 32개 중 검증 통과 32개, 응답 50건 중 통과 50건 (목표 오류 0건: 달성)

[오류 주입 결과]
 - item.qid: 형식 불일치 'Q1'
 - item.diff: 허용되지 않은 값 '4'
 - context.comm: 필수 필드 누락
 - MC: 오답 보기별 오답 유형 매핑 불완전
 - response.sid: 형식 불일치 '병사1'
 - response.submitted_s: 최대값 초과
 - response.qid: 문항 DB에 없는 문항

[문항 구성]
유형     MC  ORDER  SA  합계
영역                      
경보보고    4      1   2   7
교전통제    4      1   1   6
대드론일반   4      1   1   6
피아식별    4      1   2   7
형상식별    4      0   2   6
합계     20      4   8  32

[평가 기준]
 - 배점(난이도 1/2/3): 2 / 3 / 5점
 - 서술형: 10점(교관 채점)
 - 합격 기준: 총점의 70% 이상
 - 미응답: 0점, 미응답으로 기록(삭제 안 함)
 - 만점: 99점(서술형 제외), 오답 유형: 개념 혼동, 피아 오인, 절차 누락

스키마 필드 14개 정의(문항 5, 상황 4, 응답 5), 정의 완성도 100%
문항 예시: Q04 원거리라 실루엣 식별이 어려울 때 먼저 할 행동은? | 정답 2 | 오답 유형 {1: '절차 누락', 3: '절차 누락', 4: '피아 오인'}
```
</details>
